In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../src"))   # if notebook is in notebooks/

# or use absolute path:
# sys.path.append("C:/Users/ROB8087/.../Athlelorien/src")




import pandas as pd
import s3fs
import os
import pandas as pd
from dotenv import load_dotenv
from utils.player_utils import PlayerNameMapper
from utils.day_loader import DayDataLoader



load_dotenv()

fs = s3fs.S3FileSystem(
    key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret=os.getenv("AWS_SECRET_ACCESS_KEY"),
    client_kwargs={"region_name": os.getenv("AWS_DEFAULT_REGION", "eu-west-2")}
)


In [13]:
# List all files in your bucket
bucket = "ucl-ai-soccormon-dataset"
files = fs.ls(bucket)
print("First 5 files in bucket:")
for f in files[:5]:
    print(f)


First 5 files in bucket:
ucl-ai-soccormon-dataset/data


In [14]:
top = fs.ls("ucl-ai-soccormon-dataset")
print(top)

['ucl-ai-soccormon-dataset/data']


In [15]:
subdirs = fs.ls("ucl-ai-soccormon-dataset/data")
print("First few subdirs:", subdirs)

First few subdirs: ['ucl-ai-soccormon-dataset/data/', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020', 'ucl-ai-soccormon-dataset/data/objective_team_B_2020']


In [16]:
# Explore Team A
team_a = fs.ls("ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020")
print("Months for Team A:", team_a[:5])

Months for Team A: ['ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-07', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-08', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-09', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-10']


In [17]:
june = fs.ls("ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06")
print("Days in June:", june[:5])

Days in June: ['ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-02', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-03', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-04', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-05']


In [18]:
day1 = fs.ls("ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01")
print("Files:", day1)

# Load the first parquet file
df = pd.read_parquet(day1[0], filesystem=fs)
print("Shape:", df.shape)
df.head().to_clipboard()

Files: ['ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-1846d424-c17c-6279-23c6-612f48268673.parquet', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-23a7711a-8133-2876-37eb-dcd9e87a1613.parquet', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93.parquet', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-32fed4b3-d7fc-482d-ba21-c46c58f015b5.parquet', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a.parquet', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-4051bba7-1170-4c43-b912-8c38815a7625.parquet', 'ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01/2020-06-01-TeamA-5487ce1e-af19-922a-d9b8-a714e61a441c.parquet', 'ucl-ai-soccormon-dataset/data/objective

In [ ]:
from src.utils.player_utils import PlayerNameMapper
from src.utils.day_loader import DayDataLoader

# Assume fs is your s3fs filesystem
mapper = PlayerNameMapper()
loader = DayDataLoader(fs, mapper)

# Load one day
day_path = "ucl-ai-soccormon-dataset/data/objective_TEAM_A_2020/2020-06/2020-06-01"
df_day = loader.load_day(day_path)
print("Loaded shape:", df_day.shape)
print(df_day.head())

# Aggregate to 1s intervals using average
df_agg_mean = loader.aggregate_time(df_day, freq="1s", method="mean")
print("Aggregated mean:", df_agg_mean.head())


In [ ]:
# Third row (nth=2) every 5s
df_nth = loader.aggregate_time(df_day, freq="5s", method="first")

In [ ]:
import os
print(os.getcwd())   # should be Athlelorien/


In [ ]:
# Load some parquet file into df
# df = pd.read_parquet(...)

mapper = PlayerNameMapper()

# Apply mapping (auto-updates if new IDs appear)
df_named = mapper.apply_mapping(df)

print(df_named.head())


In [15]:
df.head().to_clipboard()

In [ ]:
parquet_files = [f for f in fs.ls(bucket) if f.endswith(".parquet")]

print("Found", len(parquet_files), "parquet files")
print("First 3:", parquet_files[:0])

# Load one
df = pd.read_parquet(parquet_files[0], filesystem=fs)
print("Shape:", df.shape)
df.head()

In [ ]:
import pyarrow.parquet as pq

meta = pq.ParquetFile(files[0], filesystem=fs)
print("Num rows:", meta.metadata.num_rows)
print("Num row groups:", meta.num_row_groups)
print("Schema:")
print(meta.schema)

In [ ]:
# Pick a file from the bucket
sample_file = files[0]  # you can also paste a known path
df = pd.read_parquet(files[0], filesystem=fs)
print("Shape:", df.shape)
df.head()


In [ ]:
from src.utils.data_access import S3DataAccess
s3 = S3DataAccess()
df = s3.read_parquet("ucl-ai-soccormon-dataset/path/to/file.parquet")
